In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-11 09:41:52 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Vinculaciones 1:1 Usando tabla resultados_vspc_medios_de_pago.gsap_m_comercios

Es una tabla full incremental.

Sirve para construir adecuadamente un histórico de los vinculados activos en cada mes, tanto para adquirencia como para wompi.

La columna id_comercio es el identificador de la adquirencia, usarla como varchar [no castearla a bigint]

Para identificar los vinculados 1:1 es mediante la comparación del id_comercio_padre con el nit. Si son iguales [castear a bigint ambos campos] entonces se identifica id_comercio 1:1

La última ingestión de la tabla refleja adecuadamente los vinculados activos en periodos anteriores. Por ejemplo se verífico los activos del mes 2022-01 en diferentes ingestiones, se observa que la variación es 0.02%. Variación baja que se mantiene en los eneros del resto de años. Por tanto, usar la última ingestión de la tabla para construir históricos es un camino adecuado.


In [2]:
dict_ult_ing_adqu_vinc = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_m_comercios')
dict_ult_ing_adqu_vinc

2026-08-11 09:43:34 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_m_comercios
2026-08-11 09:43:34 - [INFO] - Transcurrido: 1786459414, Tiempo de Refresco = 1000
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-08-11 09:43:43 - [INFO] - Finalizo la busqueda, duracion: 00:09.7, resultado: {'year': 2026, 'month': 8, 'day': 10}


{'year': 2026, 'month': 8, 'day': 10}

## Analisis ingestión vinculaciones

### Cantidad de datos por ingestión

In [3]:
sql = """
SELECT YEAR,
       MONTH,
       DAY,
       count(*) as frec
FROM resultados_vspc_medios_de_pago.gsap_m_comercios
WHERE YEAR BETWEEN 2018 AND 2026
AND MONTH BETWEEN 1 AND 12
AND DAY BETWEEN 1 AND 31
GROUP BY 1,2,3
ORDER BY year DESC, month DESC, day DESC;
"""
helper.obtener_dataframe(sql)

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 1/1 DATAFRAME         ejecutando   09:44:22 AM             

2026-08-11 09:45:14 - [INFO] - 1,496 filas, 4 columnas, 00:51.8 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 1/1 DATAFRAME         finalizado   09:44:22 AM     00:51.9 
------------------------------------------------------------


,year,month,day,frec
0,2026,8,10,791117
1,2026,8,6,790757
2,2026,8,5,790492
3,2026,8,4,790216
4,2026,8,3,790026
...,...,...,...,...
1491,2020,6,25,312500
1492,2020,6,24,312209
1493,2020,6,2,308499
1494,2020,4,14,302840


### ¿Cómo se ingestán las fechas de vinculación?

In [4]:

sql = """
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          concat(left(cast(f_vinculacion_opy as string), 7), '-01') as f_trx,
          count(*) AS num_registros
   FROM resultados_vspc_medios_de_pago.gsap_m_comercios
   WHERE YEAR BETWEEN 2022 AND 2026
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND UPPER(estado_comercio) LIKE 'ACTIV%%'
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_registros) OVER (PARTITION BY f_trx) AS total_compras,
                                sum(num_registros) OVER (PARTITION BY f_trx
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_registros_cumsum
   FROM outcome1
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY f_trx
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_registros /total_compras, 4) AS prop,
                         round(num_registros_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY f_trx DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
df_prueba = helper.obtener_dataframe(sql)

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 2/2 DATAFRAME        descargando   09:48:39 AM             

2026-08-11 09:53:01 - [INFO] - 60,660 filas, 10 columnas, 04:20.2 consultando, 00:01.1 descargando, 00:00.1 convirtiendo


 2/2 DATAFRAME         finalizado   09:48:39 AM     04:21.5 
------------------------------------------------------------


In [144]:
df_prueba

,year,month,day,f_trx,num_registros,total_compras,num_registros_cumsum,num_ing,prop,prop_cumsum
0,2026,3,18,2026-03-01,3301,21209,21209,13,0.1556,1.0000
1,2026,3,17,2026-03-01,3035,21209,17908,12,0.1431,0.8444
2,2026,3,16,2026-03-01,2713,21209,14873,11,0.1279,0.7013
3,2026,3,13,2026-03-01,2447,21209,12160,10,0.1154,0.5733
4,2026,3,12,2026-03-01,2226,21209,9713,9,0.1050,0.4580
...,...,...,...,...,...,...,...,...,...,...
53141,2022,1,6,2019-09-01,4,4112,20,5,0.0010,0.0049
53142,2022,1,5,2019-09-01,4,4112,16,4,0.0010,0.0039
53143,2022,1,4,2019-09-01,4,4112,12,3,0.0010,0.0029
53144,2022,1,3,2019-09-01,4,4112,8,2,0.0010,0.0019


In [145]:
# Transformar f_trx a formato datetime
df_prueba['f_trx'] = pd.to_datetime(df_prueba['f_trx'])
df_prueba.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53146 entries, 0 to 53145
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   year                  53146 non-null  int64         
 1   month                 53146 non-null  int64         
 2   day                   53146 non-null  int64         
 3   f_trx                 53146 non-null  datetime64[ns]
 4   num_registros         53146 non-null  int64         
 5   total_compras         53146 non-null  int64         
 6   num_registros_cumsum  53146 non-null  int64         
 7   num_ing               53146 non-null  int64         
 8   prop                  53146 non-null  float64       
 9   prop_cumsum           53146 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(7)
memory usage: 4.1 MB


In [146]:
df_prueba

,year,month,day,f_trx,num_registros,total_compras,num_registros_cumsum,num_ing,prop,prop_cumsum
0,2026,3,18,2026-03-01,3301,21209,21209,13,0.1556,1.0000
1,2026,3,17,2026-03-01,3035,21209,17908,12,0.1431,0.8444
2,2026,3,16,2026-03-01,2713,21209,14873,11,0.1279,0.7013
3,2026,3,13,2026-03-01,2447,21209,12160,10,0.1154,0.5733
4,2026,3,12,2026-03-01,2226,21209,9713,9,0.1050,0.4580
...,...,...,...,...,...,...,...,...,...,...
53141,2022,1,6,2019-09-01,4,4112,20,5,0.0010,0.0049
53142,2022,1,5,2019-09-01,4,4112,16,4,0.0010,0.0039
53143,2022,1,4,2019-09-01,4,4112,12,3,0.0010,0.0029
53144,2022,1,3,2019-09-01,4,4112,8,2,0.0010,0.0019


In [156]:
# f_trx mayores a 2026-06-01
# df_prueba[df_prueba['f_trx'] > pd.to_datetime('2026-01-01')]
df_prueba[df_prueba['f_trx'].isin(['2022-01-01'])]

,year,month,day,f_trx,num_registros,total_compras,num_registros_cumsum,num_ing,prop,prop_cumsum
25382,2026,3,18,2022-01-01,4434,4495910,4495910,1028,0.0010,1.0000
25383,2026,3,17,2022-01-01,4434,4495910,4491476,1027,0.0010,0.9990
25384,2026,3,16,2022-01-01,4434,4495910,4487042,1026,0.0010,0.9980
25385,2026,3,13,2022-01-01,4434,4495910,4482608,1025,0.0010,0.9970
25386,2026,3,12,2022-01-01,4434,4495910,4478174,1024,0.0010,0.9961
...,...,...,...,...,...,...,...,...,...,...
26405,2022,1,11,2022-01-01,373,4495910,977,5,0.0001,0.0002
26406,2022,1,7,2022-01-01,270,4495910,604,4,0.0001,0.0001
26407,2022,1,6,2022-01-01,197,4495910,334,3,0.0000,0.0001
26408,2022,1,5,2022-01-01,102,4495910,137,2,0.0000,0.0000


In [160]:
# Cantidad de vinculados en Enero 2022 en la ingestión 20220201: 4435
# Cabtudad de vubcykadis eb Enero 2022 en la ingestión 20260318: 4434

f'Porcentaje de cambio en 4 años: {round((4434-4435)/4435*100, 2)}%'

# Se mantiene un porcentaje bajo en los siguientes años'

'Porcentaje de cambio en 4 años: -0.02%'

## Obtener primera ingestión de cada mes

La ingestión de cada mes será la mejor representación de la adquirencia del mes pasado

In [6]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las ingestiones a almacenar [Esto para efectos de facilitar la actualización del histórico]
fecha_inicial = '2026-07-01' # MODIFICAR. DEBE SER EL PRIMER DÍA DE INGESTIÓN DE TRANSACCIONES A ALMACENAR O EL SIGUIENTE DÍA DESPUÉS DEL ÚLTIMO EN UNA ACTUALIZACIÓN.
fecha_final = '2026-08-31' # MODIFICAR. DEBE SER EL ÚLTIMO DÍA DE INGESTIÓN DE TRANSACCIONES ALMACENADAS O EL DÍA MÁS RECIENTE EN UNA ACTUALIZACIÓN

fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='MS')

# Ir por las particiones del siguiente mes que contendrá las vinculaciones del mes anterior
siguiente_mes = fechas[-1] + relativedelta(months=1)


# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
# # df_config['year'] = df_config['fechas'].dt.year
# # df_config['month'] = df_config['fechas'].dt.month
# # df_config['day'] = df_config['fechas'].dt.day
# df_config['sgte_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1))
df_config

,fechas
0,2026-07-01
1,2026-08-01


In [7]:
# Obtener primer día de ingestión de cada mes y almacenar en un dataframe llamado df_1_dia_ing_vinc
df_1_dia_ing_vinc = pd.DataFrame()

for index, row in df_config.iterrows():
    fecha = row['fechas']
    print('#' * 50)
    print(f"Obteniendo datos de las particiones {fecha.date().isoformat()}")
    sql = f"""
    SELECT YEAR,
           MONTH,
           DAY,
           count(*) as frec
    FROM resultados_vspc_medios_de_pago.gsap_m_comercios
    WHERE YEAR = {str(fecha.year)}
    AND MONTH = {str(fecha.month)}
    AND DAY = {str(fecha.day)}
    GROUP BY 1,2,3;
    """
    # print(sql)

    df_resultado = helper.obtener_dataframe(sql)
    print('df_resultado')
    print(df_resultado)

    # La partición puede no contener datos, por lo que se debe iterar día a día hasta encontrar la partición con datos más cercana a la fecha objetivo.
    adicionar_dias = 1
    while df_resultado.empty:
        print(f"No se encontraron datos para la fecha {fecha.date().isoformat()}. Intentando con la fecha {fecha.date().isoformat()} + {adicionar_dias} días.")
        fecha_modificada = fecha + relativedelta(days=adicionar_dias)
        sql = f"""
        SELECT YEAR,
               MONTH,
               DAY,
               count(*) as frec
        FROM resultados_vspc_medios_de_pago.gsap_m_comercios
        WHERE YEAR = {str(fecha_modificada.year)}
        AND MONTH = {str(fecha_modificada.month)}
        AND DAY = {str(fecha_modificada.day)}
        GROUP BY 1,2,3;
        """
        df_resultado = helper.obtener_dataframe(sql)
        print('Días adicionados: ', adicionar_dias)
        print(df_resultado)
        adicionar_dias += 1

    # Almacenar resultado en df_1_dia_ing_vinc    
    df_1_dia_ing_vinc = pd.concat([df_1_dia_ing_vinc, df_resultado], ignore_index=True)

df_1_dia_ing_vinc

##################################################
Obteniendo datos de las particiones 2026-07-01
------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 3/3 DATAFRAME         ejecutando   09:54:26 AM             

2026-08-11 09:55:39 - [INFO] - 1 filas, 4 columnas, 01:13.4 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 3/3 DATAFRAME         finalizado   09:54:26 AM     01:13.5 
------------------------------------------------------------
df_resultado
   year  month  day    frec
0  2026      7    1  783909
##################################################
Obteniendo datos de las particiones 2026-08-01
------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 4/4 DATAFRAME         ejecutando   09:55:39 AM             

2026-08-11 09:56:54 - [INFO] - 0 filas, 4 columnas, 01:14.2 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 4/4 DATAFRAME         finalizado   09:55:39 AM     01:14.3 
------------------------------------------------------------
df_resultado
Empty DataFrame
Columns: [year, month, day, frec]
Index: []
No se encontraron datos para la fecha 2026-08-01. Intentando con la fecha 2026-08-01 + 1 días.
------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 5/5 DATAFRAME        descargando   09:56:54 AM             

2026-08-11 09:57:22 - [INFO] - 0 filas, 4 columnas, 00:27.5 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 5/5 DATAFRAME         finalizado   09:56:54 AM     00:28.1 
------------------------------------------------------------
Días adicionados:  1
Empty DataFrame
Columns: [year, month, day, frec]
Index: []
No se encontraron datos para la fecha 2026-08-01. Intentando con la fecha 2026-08-01 + 2 días.
------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 6/6 DATAFRAME         ejecutando   09:57:22 AM             

2026-08-11 09:57:34 - [INFO] - 1 filas, 4 columnas, 00:12.4 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 6/6 DATAFRAME         finalizado   09:57:22 AM     00:12.4 
------------------------------------------------------------
Días adicionados:  2
   year  month  day    frec
0  2026      8    3  790026


,year,month,day,frec
0,2026,7,1,783909
1,2026,8,3,790026


In [ ]:
# # Escribir Histórico resultado en un archivo Excel
# df_1_dia_ing_vinc.to_excel('main_data/df_primer_dia_ing_vinc.xlsx', index=False)

In [11]:
# Actualizar histórico
df_1_dia_ing_vinc_hist = pd.read_excel('main_data/df_primer_dia_ing_vinc.xlsx')

# Unir df_ing df_1_dia_ing_vinc_hist con df_1_dia_ing_vinc
df_1_dia_ing_vinc_hist = pd.concat([df_1_dia_ing_vinc_hist, df_1_dia_ing_vinc], ignore_index=True)

# Escribir Histórico resultado en un archivo Excel
df_1_dia_ing_vinc_hist.to_excel('main_data/df_primer_dia_ing_vinc.xlsx', index=False)

df_1_dia_ing_vinc_hist

,year,month,day,frec
0,2022,1,1,456338
1,2022,2,1,460773
2,2022,3,1,466906
3,2022,4,1,474754
4,2022,5,2,481778
5,2022,6,1,488929
6,2022,7,1,496027
7,2022,8,1,502901
8,2022,9,1,510939
9,2022,10,3,519318


In [5]:
# Cantidad de clientes vinculados a corte de 2025
sql = """
SELECT count(*)
FROM resultados_vspc_medios_de_pago.gsap_m_comercios
WHERE YEAR = """ + str(dict_ult_ing_adqu_vinc['year']) + """
  AND MONTH = """ + str(dict_ult_ing_adqu_vinc['month']) + """
  AND DAY = """ + str(dict_ult_ing_adqu_vinc['day']) + """
  AND UPPER(estado_comercio) LIKE 'ACTIV%%'
  AND CASE
           WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
           ELSE 0
       END = 1;
"""
helper.obtener_dataframe(sql)

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 1/1 DATAFRAME         ejecutando   11:36:21 AM             

2026-03-24 11:36:49 - [INFO] - 1 filas, 1 columnas, 00:28.2 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 1/1 DATAFRAME         finalizado   11:36:21 AM     00:28.2 
------------------------------------------------------------


,expr_1
0,456395


In [ ]:
# Obtener información vinculados 1:1
# 1:1 se identifica cuando el nit es igual al id_comercio_padre
sql = """
SELECT YEAR,
       MONTH,
       DAY,
       id_comercio_padre,
       cast(id_comercio_padre AS BIGINT) AS id_comercio_padre_bigint,
       id_comercio,
       nit,
       f_vinculacion_opy,
       f_ini_actividad_opy,
       f_desafiliacion,
       estado_comercio,
       CASE
           WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
           ELSE 0
       END AS 1_a_1
FROM resultados_vspc_medios_de_pago.gsap_m_comercios
WHERE YEAR BETWEEN 2026 AND 2026
  AND MONTH BETWEEN 3 AND 3
  AND DAY BETWEEN 13 AND 13
  AND UPPER(estado_comercio) LIKE 'ACTIV%%';
"""
helper.obtener_dataframe(sql)

2026-03-16 16:38:45 - [INFO] - Transcurrido: 1773697125, Tiempo de Refresco = 1000


------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 1/1 DATAFRAME          falló n.1   04:38:45 PM             
 1/1 DATAFRAME          falló n.2   04:38:45 PM             
 1/1 DATAFRAME              error   04:38:45 PM     05:08.0 
------------------------------------------------------------

------------------------------------------------------------
Consulta que fallo:

SELECT YEAR,
       MONTH,
       DAY,
       id_comercio_padre,
       cast(id_comercio_padre AS BIGINT) AS id_comercio_padre_bigint,
       id_comercio,
       nit,
       f_vinculacion_opy,
       f_ini_actividad_opy,
       f_desafiliacion,
       estado_comercio,
       CASE
           WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
           ELSE 0
       END AS 1_a_1
FROM resultados_vspc_medios_de_pago.gsap_m_comercios
WHERE YEAR BETWEEN 2026 AND 2026
  AND

Error: ('HY000', '[HY000] [Cloudera][ImpalaODBC] (110) Error while executing a query in Impala: [HY000] : Runtime Error: Failed to get minimum memory reservation of 4.00 MB on daemon sbmdeblzw052.bancolombia.corp:27000 for query ae4669665826ec71:bfa9b87500000000 due to following error: Memory limit exceeded: Could not allocate memory while trying to increase reservation.\nQuery(ae4669665826ec71:bfa9b87500000000) could not allocate 4.00 MB without exceeding limit.\nError occurred on backend sbmdeblzw052.bancolombia.corp:27000\nMemory l\x00 (110) (SQLExecDirectW)')

In [4]:
sql = """DESCRIBE resultados_vspc_medios_de_pago.gsap_m_comercios;"""
df_temp = helper.obtener_dataframe(sql)
df_temp

-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 1/1 DATAFRAME ..._vspc_medios_de_pago.gsap_m_comercios  descargando   02:37:02 PM             

2026-03-16 14:37:05 - [INFO] - 124 filas, 3 columnas, 00:01.6 consultando, 00:00.8 descargando, 00:00.0 convirtiendo


 1/1 DATAFRAME ..._vspc_medios_de_pago.gsap_m_comercios   finalizado   02:37:02 PM     00:02.9 
-----------------------------------------------------------------------------------------------


,name,type,comment
0,institucion,varchar(3),Institución
1,nivel_comercio,varchar(1),Nivel del comercio en la jerarquia de Omnipay
2,id_comercio,varchar(20),"Id externo del comercio, a nivel de miembro es..."
3,num_comercio_opy,varchar(8),Id interno de Omnipay para el comercio
4,nombre_comercio,varchar(35),Nombre del comercio
...,...,...,...
119,ingestion_month,tinyint,Mes de ingestión
120,ingestion_day,tinyint,Día de ingestión
121,year,smallint,Año de partición
122,month,tinyint,Mes de partición


In [5]:
nombre_cols_string = ', '.join(df_temp['name'].tolist())
nombre_cols_string

'institucion, nivel_comercio, id_comercio, num_comercio_opy, nombre_comercio, razon_social, id_comercio_padre, nit, tipo_sociedad, cod_mcc, descri_mcc, nombre_contacto, idioma_comercio, region, estado_comercio, f_vinculacion_opy, f_ini_actividad_opy, f_desafiliacion, grado_comercio, id_contrato_servicio, cod_rcc, descri_rcc, cod_pais, descri_pais, descri_ciudad, cod_dpto, descri_dpto, cod_municipio, descri_municipio, indicador_reteica, cod_retefuente, descri_retefuente, cod_reteiva, descri_reteiva, indicador_propina, cod_estado_red_mc, cod_estado_red_visa, cod_estado_red_amex, dir_ppal, f_aplicacion_dir_ppal, contacto_dir_ppal, ciudad_dir_ppal, tel_dir_ppal, email_dir_ppal, dir_ccial, f_aplicacion_dir_ccial, contacto_dir_ccial, ciudad_dir_ccial, tel_dir_ccial, email_dir_ccial, dir_legal, f_aplicacion_dir_legal, contacto_dir_legal, ciudad_dir_legal, tel_dir_legal, email_dir_legal, dir_disp, f_aplicacion_dir_disp, contacto_dir_disp, ciudad_dir_disp, tel_dir_disp, email_dir_disp, llave_md

In [21]:
nombre_cols_list = df_temp['name'].tolist()
string_where_filtro = ''
for col in nombre_cols_list:
    string_where_filtro += f" AND LOWER(TRIM(cast({col} as string))) LIKE '%wompi%'"
string_where_filtro


" AND LOWER(TRIM(cast(institucion as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(nivel_comercio as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(id_comercio as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(num_comercio_opy as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(nombre_comercio as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(razon_social as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(id_comercio_padre as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(nit as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(tipo_sociedad as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(cod_mcc as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(descri_mcc as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(nombre_contacto as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(idioma_comercio as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(region as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(estado_comercio as string))) LIKE '%wompi%' AND LOWER(TRIM(cast(f_vinculacion_opy as string))) LIKE '%wompi%' AND LOWER(TRIM(ca

In [22]:
sql = """
select """ + nombre_cols_string + """
from resultados_vspc_medios_de_pago.gsap_m_comercios
WHERE YEAR = """ + str(dict_ult_ing_adqu_vinc['year']) + """
AND MONTH = """ + str(dict_ult_ing_adqu_vinc['month']) + """
AND DAY = """ + str(dict_ult_ing_adqu_vinc['day']) + """
AND """ + string_where_filtro + """;"""
sql

"\nselect institucion, nivel_comercio, id_comercio, num_comercio_opy, nombre_comercio, razon_social, id_comercio_padre, nit, tipo_sociedad, cod_mcc, descri_mcc, nombre_contacto, idioma_comercio, region, estado_comercio, f_vinculacion_opy, f_ini_actividad_opy, f_desafiliacion, grado_comercio, id_contrato_servicio, cod_rcc, descri_rcc, cod_pais, descri_pais, descri_ciudad, cod_dpto, descri_dpto, cod_municipio, descri_municipio, indicador_reteica, cod_retefuente, descri_retefuente, cod_reteiva, descri_reteiva, indicador_propina, cod_estado_red_mc, cod_estado_red_visa, cod_estado_red_amex, dir_ppal, f_aplicacion_dir_ppal, contacto_dir_ppal, ciudad_dir_ppal, tel_dir_ppal, email_dir_ppal, dir_ccial, f_aplicacion_dir_ccial, contacto_dir_ccial, ciudad_dir_ccial, tel_dir_ccial, email_dir_ccial, dir_legal, f_aplicacion_dir_legal, contacto_dir_legal, ciudad_dir_legal, tel_dir_legal, email_dir_legal, dir_disp, f_aplicacion_dir_disp, contacto_dir_disp, ciudad_dir_disp, tel_dir_disp, email_dir_disp,

In [23]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_adqu PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_adqu STORED AS PARQUET AS
SELECT id_comercio_padre,
          id_comercio AS cod_unico,
          f_vinculacion_opy 

   FROM resultados_vspc_medios_de_pago.gsap_m_comercios
   WHERE YEAR = """ + str(dict_ult_ing_adqu_vinc['year']) + """
   AND MONTH = """ + str(dict_ult_ing_adqu_vinc['month']) + """
   AND DAY = """ + str(dict_ult_ing_adqu_vinc['day']) + """
   AND (id_comercio_padre = '16420697'
          OR id_comercio_padre = '16249062'
          OR id_comercio_padre = '16243073'
          OR id_comercio_padre = '15021702'
          OR id_comercio_padre = '15026362'
          OR id_comercio_padre = '19245380'
          OR id_comercio_padre = '19747609'
          OR id_comercio_padre = '19149806'
          OR id_comercio_padre = '20657045'
          OR id_comercio_padre = '20750808'
          OR id_comercio_padre = '18965848'
          OR id_comercio_padre = '22498349'
          OR id_comercio_padre = '19969146'
          OR id_comercio_padre = '22701601'
          OR id_comercio_padre = '22845325'
          OR id_comercio_padre = '22784474'
          OR id_comercio_padre = '22756944'
          OR id_comercio_padre = '22756902'
          OR id_comercio_padre = '22837413'
          OR id_comercio_padre = '22831580')
     AND UPPER(estado_comercio) LIKE 'ACTIV%%';
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE STATS proceso.mdo_aceptacion_comercios_hist_vinc_adqu;"""
helper.ejecutar_consulta(sql_compute)

2026-02-05 16:09:44 - [INFO] - Transcurrido: 2823, Tiempo de Refresco = 1000


-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 3/3      DROP ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   04:09:46 PM     00:00.4 
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 4/4    CREATE ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   04:09:47 PM     00:03.0 
-----------------------------------------------------------------------------------------------
----------------------------------------

# Vinculaciones usando tabla resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf

## Supuesto 1: Vinculación por primera vez

Se asume que la primera vinculación es el la vinculación

In [ ]:
dict_ult_ing_adqu_vinc = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf')
dict_ult_ing_adqu_vinc

In [ ]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_adqu PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_adqu STORED AS PARQUET AS WITH vinculados AS
  (SELECT codigo_unico,
          min(periodo) AS fecha_ym
   FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
   WHERE YEAR <= """ + str(dict_ult_ing_adqu_vinc['year']) + """
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND periodo IS NOT NULL
   GROUP BY 1),
                                                                                       conteo AS
  (SELECT fecha_ym,
          count(*) AS num_vinc_new,
          cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes
   FROM vinculados
   GROUP BY 1)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
                          'adquirencia' AS producto
FROM conteo
ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_adqu;"""
helper.ejecutar_consulta(sql_compute)

## Supuesto 2: La vinculación es dinámica en el tiempo

Un cliente se puede vincular, luego desvincularse y finalmente volverse a vincular

In [ ]:
dict_ult_ing_adqu_vinc = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf')
dict_ult_ing_adqu_vinc

In [ ]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_adqu PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_adqu STORED AS PARQUET AS WITH vinculados AS
  (SELECT codigo_unico,
          extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'year')*100 + extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'month') AS fecha_ym
      FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
      WHERE YEAR <= """ + str(dict_ult_ing_adqu_vinc['year']) + """
      AND MONTH BETWEEN 1 AND 12
      AND DAY BETWEEN 1 AND 31),
                                                                                       conteo AS
  (SELECT fecha_ym,
          count(*) AS num_vinc_new,
          cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes
   FROM vinculados
   GROUP BY 1)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
                          'adquirencia' AS producto
FROM conteo
ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_adqu;"""
helper.ejecutar_consulta(sql_compute)

## 

# Borrar tablas proceso.

In [4]:
# Borrar tablas proceso.
tablas_borrar = ['proceso.mdo_aceptacion_comercios_hist_vinc_adqu']
for tabla in tablas_borrar:
    sql_drop = f"""DROP TABLE IF EXISTS {tabla} PURGE;"""
    helper.ejecutar_consulta(sql_drop)

2026-02-05 14:26:33 - [INFO] - Transcurrido: 1170, Tiempo de Refresco = 1000


------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 1/1 DROP ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   02:26:34 PM     00:00.5 
------------------------------------------------------------------------------------------
